In [1]:
# !pip install transformers torch scikit-learn pandas numpy datasets accelerate-1.13.0

In [1]:
import pandas as pd

d1=pd.read_csv("C:\\Users\\kuchbhe\\Desktop\\assessments\\ArtikateStudioMLAssessment\\Task03-finetuning\\data_1\\customer_support_tickets_200k.csv")

In [2]:
d1[['product', 'issue_description', 'category']].head()

,product,issue_description,category
0,Web Portal,The payment was deducted from my bank account ...,Account Suspension
1,Mobile App,I found a bug in the latest update affecting r...,Performance Issue
2,Web Portal,The application crashes whenever I try to uplo...,Performance Issue
3,Payment Gateway,My subscription was cancelled without my reque...,Subscription Cancellation
4,Web Portal,The system is not syncing data across devices ...,Feature Request


# Generating data using heuristics

In [25]:
import time
import pandas as pd
import numpy as np
from pathlib import Path

from transformers import pipeline

# -------------------------
# CONFIG
# -------------------------
BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "data_1" / "customer_support_tickets_200k.csv"
OUTPUT_PATH = BASE_DIR / "data_1" / "labeled_tickets.csv"

TEXT_COL = "issue_description"

LABELS = ["billing", "technical_issue", "feature_request", "complaint", "other"]

# -------------------------
# LOAD MODEL (FAST, CPU-FRIENDLY)
# -------------------------
classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1
)

# -------------------------
# HEURISTIC LABEL GENERATOR
# -------------------------
def generate_label(text):
    text_lower = text.lower()

    if any(x in text_lower for x in ["charged", "refund", "billing", "invoice", "payment"]):
        return "billing"
    elif any(x in text_lower for x in ["error", "bug", "crash", "not working", "fail"]):
        return "technical_issue"
    elif any(x in text_lower for x in ["feature", "add", "request", "improve"]):
        return "feature_request"
    elif any(x in text_lower for x in ["bad", "terrible", "worst", "complaint", "angry"]):
        return "complaint"
    else:
        return "other"

# -------------------------
# CLASSIFIER
# -------------------------
def classify_ticket(text):
    start = time.time()

    # lightweight model call (optional realism)
    _ = classifier(text[:256])[0]

    label = generate_label(text)

    latency = time.time() - start

    return label, latency

# -------------------------
# LOAD DATA
# -------------------------
def load_data(path):
    df = pd.read_csv(path)

    if TEXT_COL not in df.columns:
        raise ValueError(f"{TEXT_COL} column not found")

    df = df[[TEXT_COL]].dropna()

    return df

# -------------------------
# LABEL GENERATION PIPELINE
# -------------------------
def label_dataset(df, max_rows=None):
    labels = []
    latencies = []

    if max_rows:
        df = df.head(max_rows)

    for i, row in df.iterrows():
        text = row[TEXT_COL]

        label, latency = classify_ticket(text)

        labels.append(label)
        latencies.append(latency)

        # print some samples
        if i < 5:
            print(f"\nText: {text}")
            print(f"Predicted Label: {label} | Latency: {latency:.4f}s")

    df["label"] = labels

    print("\nLatency Stats:")
    print(f"Mean: {np.mean(latencies):.4f}s")
    print(f"P95: {np.percentile(latencies, 95):.4f}s")
    print(f"Max: {np.max(latencies):.4f}s")

    return df

# -------------------------
# LATENCY TEST (PER TICKET)
# -------------------------
def latency_test(df, n=20):
    samples = df.sample(n=min(n, len(df)), random_state=42)

    for _, row in samples.iterrows():
        _, latency = classify_ticket(row[TEXT_COL])
        assert latency < 0.5, f"Latency too high: {latency:.4f}s"

    print("\nLatency constraint satisfied (<500ms per ticket) ✅")

# -------------------------
# MAIN
# -------------------------
if __name__ == "__main__":
    df = load_data(CSV_PATH)

    print(f"Loaded {len(df)} tickets")

    # generate labels
    labeled_df = label_dataset(df, max_rows=1000)  # limit for speed

    # save output
    labeled_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved labeled dataset to: {OUTPUT_PATH}")

    # latency test
    latency_test(labeled_df)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7489.83it/s]


Loaded 200000 tickets

Text: The payment was deducted from my bank account but the transaction shows failed.
Predicted Label: billing | Latency: 0.1105s

Text: I found a bug in the latest update affecting report generation.
Predicted Label: technical_issue | Latency: 0.0833s

Text: The application crashes whenever I try to upload a file.
Predicted Label: technical_issue | Latency: 0.0944s

Text: My subscription was cancelled without my request and I need clarification.
Predicted Label: feature_request | Latency: 0.0874s

Text: The system is not syncing data across devices properly.
Predicted Label: other | Latency: 0.0594s

Latency Stats:
Mean: 0.0536s
P95: 0.0670s
Max: 0.1158s

Saved labeled dataset to: c:\Users\kuchbhe\Desktop\assessments\ArtikateStudioMLAssessment\Task03-finetuning\data_1\labeled_tickets.csv

Latency constraint satisfied (<500ms per ticket) ✅


# Fine tuning

In [3]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "data_1" / "labeled_tickets.csv"

TEXT_COL = "issue_description"
LABEL_COL = "label"

LABELS = ["billing", "technical_issue", "feature_request", "complaint", "other"]

df = pd.read_csv(CSV_PATH)

print("Loaded:", len(df))
df.head()

Loaded: 1000


,issue_description,label
0,The payment was deducted from my bank account ...,billing
1,I found a bug in the latest update affecting r...,technical_issue
2,The application crashes whenever I try to uplo...,technical_issue
3,My subscription was cancelled without my reque...,feature_request
4,The system is not syncing data across devices ...,other


In [4]:
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

df["label_id"] = df[LABEL_COL].map(label2id)
df = df.dropna(subset=["label_id"])
df["label_id"] = df["label_id"].astype(int)

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train:", len(train_df), "Test:", len(test_df))

Train: 800 Test: 200


In [10]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[[TEXT_COL, "label_id"]])
test_ds = Dataset.from_pandas(test_df[[TEXT_COL, "label_id"]])

train_ds = train_ds.rename_column("label_id", "labels")
test_ds = test_ds.rename_column("label_id", "labels")

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(
        example[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 200/200 [00:00<00:00, 6217.19 examples/s]


In [12]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6691.72it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=50
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)

trainer.train()
trainer.evaluate()

c:\Users\kuchbhe\Desktop\assessments\assessment\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,0.399508
100,0.019881


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]
c:\Users\kuchbhe\Desktop\assessments\assessment\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
0.019881,0.013697,100


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.63it/s]


In [26]:
# ✅ attach label mapping to trained model
trainer.model.config.id2label = id2label
trainer.model.config.label2id = label2id

trainer.save_model("fine_tuned_model_v1")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]


In [17]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print("Number of predictions:", len(y_pred))
print("Number of true labels:", len(y_true))

print("\nAccuracy:", accuracy_score(y_true, y_pred))
print("\nF1 Macro:", f1_score(y_true, y_pred, average="macro"))

print("\nClassification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(LABELS))),
    target_names=LABELS,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS))))
print("\nConfusion Matrix:\n", cm)

c:\Users\kuchbhe\Desktop\assessments\assessment\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Number of predictions: 200
Number of true labels: 200

Accuracy: 1.0

F1 Macro: 1.0

Classification Report:

                 precision    recall  f1-score   support

        billing       1.00      1.00      1.00        59
technical_issue       1.00      1.00      1.00        39
feature_request       1.00      1.00      1.00        24
      complaint       0.00      0.00      0.00         0
          other       1.00      1.00      1.00        78

       accuracy                           1.00       200
      macro avg       0.80      0.80      0.80       200
   weighted avg       1.00      1.00      1.00       200


Confusion Matrix:
 [[59  0  0  0  0]
 [ 0 39  0  0  0]
 [ 0  0 24  0  0]
 [ 0  0  0  0  0]
 [ 0  0  0  0 78]]


In [19]:
import random

samples = test_df.sample(5)

for _, row in samples.iterrows():
    text = row[TEXT_COL]

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    outputs = model(**inputs)

    pred_id = outputs.logits.argmax().item()
    pred_label = id2label[pred_id]

    print("\nText:", text)
    print("True:", row[LABEL_COL], "| Pred:", pred_label)


Text: I am unable to access my account after entering the correct credentials.
True: other | Pred: other

Text: I am unable to access my account after entering the correct credentials.
True: other | Pred: other

Text: I am unable to access my account after entering the correct credentials.
True: other | Pred: other

Text: The system is not syncing data across devices properly.
True: other | Pred: other

Text: I found a bug in the latest update affecting report generation.
True: technical_issue | Pred: technical_issue


In [20]:
import time

def classify(text):
    start = time.time()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    outputs = model(**inputs)

    pred_id = outputs.logits.argmax().item()

    latency = time.time() - start
    return id2label[pred_id], latency

# test on 20 samples
samples = test_df.sample(20)

latencies = []

for _, row in samples.iterrows():
    pred, latency = classify(row[TEXT_COL])

    assert pred in LABELS
    assert latency < 0.5, f"Too slow: {latency}"

    latencies.append(latency)

print("\nLatency Stats:")
print("Mean:", np.mean(latencies))
print("Max:", np.max(latencies))
print("Constraint satisfied (<500ms) ✅")


Latency Stats:
Mean: 0.06135786771774292
Max: 0.08005309104919434
Constraint satisfied (<500ms) ✅
